# Notebook 19 — PURE HTR Fusion (CRNN + TrOCR, NO lexicon)

In [1]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import difflib

DEVICE = (torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
print(f"Using device: {DEVICE}")

DATA_ROOT = Path("../data/pharmacy_lk")
TRAIN_CSV = DATA_ROOT / "splits/train.csv"
TEST_CSV  = DATA_ROOT / "splits/test.csv"
VAL_CSV   = DATA_ROOT / "splits/val.csv"
LABEL_COL = "medicine_name"; IMG_COL = "image_filename"
CRNN_CKPT = Path("../checkpoints/baseline_crnn_v2/best.pt")
TROCR_CKPT = Path("../checkpoints/benchmark_trocr/best")
PROC = "microsoft/trocr-small-handwritten"
IMG_H, IMG_W = 48, 320

Using device: mps


## 1. Metrics

In [2]:
def edit_distance(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j]+1, cur[j-1]+1, prev[j-1]+(ca != cb)))
        prev = cur
    return prev[-1]

def metrics(preds, refs):
    tot = sum(edit_distance(p, r) for p, r in zip(preds, refs))
    chars = sum(len(r) for r in refs)
    em = sum(p == r for p, r in zip(preds, refs))
    return {"CER": tot/max(chars,1), "EM": em/len(refs), "n": len(refs)}

## 2. CRNN (raw, no lexicon) with confidence

In [3]:
class Vocab:
    BLANK = 0
    def __init__(self, texts):
        chars = sorted(set("".join(texts)))
        self.idx2char = {i+1: c for i, c in enumerate(chars)}
        self.char2idx = {c: i for i, c in self.idx2char.items()}
    def __len__(self): return len(self.idx2char)+1
    def decode_conf(self, lp_row):
        probs = lp_row.exp(); idx = lp_row.argmax(-1).tolist()
        out, prev, confs = [], None, []
        for t, i in enumerate(idx):
            if i != prev and i != self.BLANK:
                out.append(self.idx2char[i]); confs.append(float(probs[t, i]))
            prev = i
        return "".join(out), (float(np.mean(confs)) if confs else 0.0)

class CRNN(nn.Module):
    def __init__(self, n, h=256, l=2, d=0.2):
        super().__init__()
        def cv(i,o,bn=False):
            L=[nn.Conv2d(i,o,3,1,1)]
            if bn: L.append(nn.BatchNorm2d(o))
            L.append(nn.ReLU(inplace=True)); return L
        self.cnn=nn.Sequential(*cv(1,64),nn.MaxPool2d(2,2),*cv(64,128),nn.MaxPool2d(2,2),
            *cv(128,256),*cv(256,256),nn.MaxPool2d((2,1),(2,1)),
            *cv(256,512,bn=True),*cv(512,512,bn=True),nn.MaxPool2d((2,1),(2,1)))
        self.collapse=nn.AdaptiveAvgPool2d((1,None))
        self.rnn=nn.LSTM(512,h,l,bidirectional=True,dropout=d if l>1 else 0.0)
        self.head=nn.Linear(2*h,n)
    def forward(self,x):
        f=self.collapse(self.cnn(x)).squeeze(2).permute(2,0,1)
        s,_=self.rnn(f); return self.head(s)

_tr = pd.read_csv(TRAIN_CSV)[LABEL_COL].dropna().astype(str).str.strip().tolist()
VOCAB = Vocab(_tr)
ck = torch.load(CRNN_CKPT, map_location="cpu")
assert len(VOCAB)==ck["model"]["head.bias"].shape[0], "vocab mismatch"
crnn = CRNN(len(VOCAB)); crnn.load_state_dict(ck["model"]); crnn.to(DEVICE).eval()

def crnn_one(pil):
    img=pil.convert("L"); w,h=img.size
    nw=min(max(1,int(round(w*IMG_H/h))),IMG_W)
    img=img.resize((nw,IMG_H),Image.BILINEAR)
    cv=Image.new("L",(IMG_W,IMG_H),255); cv.paste(img,(0,0))
    x=torch.from_numpy(np.array(cv,dtype=np.float32)/255.0).unsqueeze(0).unsqueeze(0)
    with torch.no_grad():
        lp=crnn(x.to(DEVICE)).log_softmax(-1).permute(1,0,2)[0].cpu()
    return VOCAB.decode_conf(lp)

## 3. TrOCR (raw, no lexicon) with confidence

In [4]:
processor = TrOCRProcessor.from_pretrained(PROC)
trocr = VisionEncoderDecoderModel.from_pretrained(TROCR_CKPT).to(DEVICE).eval()

def trocr_one(pil):
    pv = processor(pil.convert("RGB"), return_tensors="pt").pixel_values.to(DEVICE)
    with torch.no_grad():
        out = trocr.generate(pv, max_new_tokens=24, output_scores=True, return_dict_in_generate=True)
    ts = trocr.compute_transition_scores(out.sequences, out.scores, normalize_logits=True)
    text = processor.decode(out.sequences[0], skip_special_tokens=True).strip().lower()
    valid = ts[0][torch.isfinite(ts[0])]
    conf = float(valid.mean().exp()) if valid.numel() else 0.0
    return text, conf

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

## 4. Collect predictions on a split

In [5]:
def collect(csv):
    df = pd.read_csv(csv).dropna(subset=[LABEL_COL])
    rows = []
    for _, r in df.iterrows():
        fn = str(r[IMG_COL]); ref = str(r[LABEL_COL]).strip().lower()
        p = DATA_ROOT / "images" / fn
        if not p.exists(): continue
        pil = Image.open(p)
        c_txt, c_conf = crnn_one(pil)
        t_txt, t_conf = trocr_one(pil)
        rows.append({"f": fn, "ref": ref, "c_txt": c_txt, "c_conf": c_conf,
                     "t_txt": t_txt, "t_conf": t_conf})
    return pd.DataFrame(rows)

print("collecting test predictions (no lexicon)...")
test = collect(TEST_CSV)
print(f"collected {len(test)} test rows")

collecting test predictions (no lexicon)...
collected 791 test rows


## 5. Lexicon-free fusion referees

In [6]:
def fuse_conf_max(row):
    return row["t_txt"] if row["t_conf"] >= row["c_conf"] else row["c_txt"]

def fuse_agreement(row):
    if row["c_txt"] == row["t_txt"]:
        return row["t_txt"]
    return row["t_txt"] if row["t_conf"] >= row["c_conf"] else row["c_txt"]

def fuse_char_vote(row):
    a, b = row["c_txt"], row["t_txt"]
    if a == b: return a
    # align and vote per character; tie -> higher-confidence model's char
    sm = difflib.SequenceMatcher(None, a, b)
    out = []
    prefer = b if row["t_conf"] >= row["c_conf"] else a
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            out.append(a[i1:i2])
        else:
            # disagreement region: take the preferred model's span
            out.append(prefer[(j1 if prefer is b else i1):(j2 if prefer is b else i2)])
    return "".join(out)

## 6. Evaluate everything

In [7]:
refs = test["ref"].tolist()
crnn_only = test["c_txt"].tolist()
trocr_only = test["t_txt"].tolist()
f_cmax = test.apply(fuse_conf_max, axis=1).tolist()
f_agree = test.apply(fuse_agreement, axis=1).tolist()
f_vote = test.apply(fuse_char_vote, axis=1).tolist()

print("PURE HTR (no lexicon) — TEST:")
for name, preds in [("CRNN only", crnn_only), ("TrOCR only", trocr_only),
                    ("Fusion: confidence-max", f_cmax),
                    ("Fusion: agreement-gated", f_agree),
                    ("Fusion: char-vote", f_vote)]:
    m = metrics(preds, refs)
    print(f"  {name:26s}: CER {m['CER']:.4f} | EM {m['EM']:.4f}")

agree = (test["c_txt"] == test["t_txt"]).mean()
print(f"\nCRNN/TrOCR raw agreement: {agree:.1%}")
print(f"mean conf — CRNN {test['c_conf'].mean():.3f} | TrOCR {test['t_conf'].mean():.3f}")

PURE HTR (no lexicon) — TEST:
  CRNN only                 : CER 0.5130 | EM 0.0392
  TrOCR only                : CER 0.2159 | EM 0.5689
  Fusion: confidence-max    : CER 0.2131 | EM 0.5676
  Fusion: agreement-gated   : CER 0.2131 | EM 0.5676
  Fusion: char-vote         : CER 0.2131 | EM 0.5676

CRNN/TrOCR raw agreement: 3.7%
mean conf — CRNN 0.583 | TrOCR 0.860


## 7. Honest interpretation
- If NO fusion variant beats TrOCR-only CER/EM: the honest finding is that TrOCR dominates and
  pure-recognition fusion offers no benefit (the weaker CRNN cannot help the stronger TrOCR
  without domain knowledge). Phase-2 pure-HTR result = TrOCR fine-tuned.
- If a fusion variant DOES beat TrOCR-only: that variant is a genuine pure-HTR novelty.
- Either way: report the agreement rate + confidence scales as the explanation.